# Walking-skeleton API demo

Runs each public entrypoint in `lr_reduction.api` one at a time, against a real YAML
configuration loaded through `ConfigLoader`. Nothing is mocked, but most of what runs
underneath is still a stub:

- `RunLoader` returns a placeholder event workspace; no NeXus file is read.
- The operations return empty curves, so every result has 0 points.
- `write_orso` writes placeholder data to `PLACEHOLDER_FILENAME.ort`.

The point is to see the call shape of each entrypoint, where it looks for its
configuration, and what it writes.

## Setup

In [ ]:
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

from mantid.kernel import ConfigService, Int32TimeSeriesProperty
from mantid.simpleapi import CreateSampleWorkspace

from lr_reduction.api import reduce_and_combine_runs, reduce_auto, reduce_batch, reduce_live, reduce_run
from lr_reduction.models.results import CombinedReductionResult, ReductionResult

# Silence Mantid's per-algorithm notices; lr_reduction's own INFO logging still shows each step.
ConfigService.setLogLevel(4)  # warning

WORKDIR = Path(tempfile.mkdtemp(prefix="lr_skeleton_demo_"))
print(f"Outputs go to {WORKDIR}")

One configuration serves every entrypoint: three reflected runs (sequence numbers 1-3)
sharing two direct-beam sets. It mirrors `tests/data/config_example.yaml`.

In [ ]:
CONFIG_YAML = """\
instrument: BL4B

defaults:
  q_binning:
    q_min: 0.005
    q_max: 0.5
    q_step: 0.01
    method: constantQ

direct_beams:
  db_8mm:
    direct_beam_run_numbers: [12345, 12346, 12347]
  db_20mm:
    direct_beam_run_numbers: [12350, 12351]

runs:
  - sequence_number: 1
    direct_beam: db_8mm
    run_number: 12360
    peak: { min: 140, max: 150 }
  - sequence_number: 2
    direct_beam: db_20mm
    run_number: 12361
    peak: { min: 138, max: 152 }
  - sequence_number: 3
    direct_beam: db_8mm
    run_number: 12362
"""
REFLECTED_RUNS = [12360, 12361, 12362]


def demo_dir(name: str) -> Path:
    """A fresh subdirectory of WORKDIR for one entrypoint."""
    path = WORKDIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_config(path: Path) -> Path:
    path.write_text(CONFIG_YAML)
    return path


def describe(result: ReductionResult | CombinedReductionResult) -> None:
    print(f"type:            {type(result).__name__}")
    print(f"curve points:    {len(result.curve.q)}")
    if isinstance(result, ReductionResult):
        print(f"run_numbers:     {result.run_numbers}")
        print(f"sequence_id:     {result.sequence_id}")
        print(f"sequence_number: {result.sequence_number}")
    else:
        print(f"partials:        {len(result.partials)}")
        print(f"scale_factors:   {result.scale_factors}")
        print(f"html_report:     {'yes' if result.html_report else 'no'}")
    print(f"config runs:     {sorted(result.reduction_config.runs)}")


def list_outputs(directory: Path) -> None:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            print(f"  {path.relative_to(directory)}")


@contextmanager
def working_directory(path: Path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

## 1. `reduce_run`: one run, explicit configuration

Takes a run number and a configuration path.

In [ ]:
workdir = demo_dir("manual")
config = write_config(workdir / "config.yaml")

result = reduce_run(REFLECTED_RUNS[0], config, output_dir=str(workdir))
describe(result)
list_outputs(workdir)

## 2. `reduce_and_combine_runs`: in-memory sequence assembly

Reduces every listed run, then combines them into one curve without going through disk.
Only the combined file is written.

In [ ]:
workdir = demo_dir("combine")
config = write_config(workdir / "config.yaml")

result = reduce_and_combine_runs(REFLECTED_RUNS, config, output_dir=str(workdir))
describe(result)
list_outputs(workdir)

## 3. `reduce_auto`: autoreduction (§6.4.1)

Called by the autoreduction service with a NeXus path and an output directory. The
configuration is `config_0.yaml` in the output directory. It reduces the new run,
writes its partial, then rebuilds the sequence from the partials on disk (§6.4.4).

`RunLoader.load_from_path` is still a stub, so an empty file named like a NeXus file is enough.

In [ ]:
workdir = demo_dir("auto")
write_config(workdir / "config_0.yaml")
nexus_file = workdir / f"REF_L_{REFLECTED_RUNS[0]}.nxs.h5"
nexus_file.touch()

result = reduce_auto(nexus_file, workdir)
describe(result)
list_outputs(workdir)

## 4. `reduce_live`: live reduction (§6.4.5)

The live-data service passes in an in-memory event workspace instead of a file. The
run number and `sequence_number` come from its sample logs. The configuration is
`run_<run_number>.yaml`, resolved relative to the current directory (placeholder
convention), and both the single-run step and the sequence assembly use it.

The workspace here is a sample event workspace with the two logs the DAS would
record. `sequence_number` is a constant time series, as on the instrument.

In [ ]:
workdir = demo_dir("live")
run_number = REFLECTED_RUNS[0]
write_config(workdir / f"run_{run_number}.yaml")

ws = CreateSampleWorkspace(WorkspaceType="Event", OutputWorkspace="skeleton_live_run")
ws.getRun().addProperty("run_number", str(run_number), True)
sequence_number = Int32TimeSeriesProperty("sequence_number")
sequence_number.addValue("2026-01-01T00:00:00", 1)
sequence_number.addValue("2026-01-01T00:00:01", 1)
ws.getRun().addProperty("sequence_number", sequence_number, True)

# The configuration lookup and the default output directory are both relative to the cwd.
with working_directory(workdir):
    result = reduce_live(ws)
describe(result)
list_outputs(workdir)

## Inspect an ORSO output

In [ ]:
print((WORKDIR / "auto" / "PLACEHOLDER_FILENAME.ort").read_text())

## Clean up

In [ ]:
import shutil

shutil.rmtree(WORKDIR)